In [1]:
import os
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
class CreditFeaturesSchema(BaseModel):
    loan_amnt: float = Field(description="The requested loan amount")
    term: str = Field(description="Loan term, strictly format as ' 36 months' or ' 60 months'")
    int_rate: float = Field(description="Interest rate on the loan")
    installment: float = Field(description="Monthly payment owed by the borrower")
    grade: str = Field(description="Assigned loan grade (e.g., A, B, C)")
    sub_grade: str = Field(description="Assigned loan sub-grade (e.g., B4)")
    emp_length: str = Field(description="Employment length in years, e.g. '10+ years'")
    home_ownership: str = Field(description="Home ownership status (RENT, OWN, MORTGAGE)")
    annual_inc: float = Field(description="Self-reported annual income")
    verification_status: str = Field(description="Income verification status")
    purpose: str = Field(description="Category provided by the borrower for the loan request")
    addr_state: str = Field(description="State address, e.g., 'CA'")
    dti: float = Field(description="Debt-to-income ratio")
    open_acc: float = Field(description="Number of open credit lines")
    pub_rec: float = Field(description="Number of derogatory public records")
    revol_bal: float = Field(description="Total credit revolving balance")
    revol_util: float = Field(description="Revolving line utilization rate")
    total_acc: float = Field(description="Total number of credit lines")
    mort_acc: float = Field(description="Number of mortgage accounts")
    pub_rec_bankruptcies: float = Field(description="Number of public record bankruptcies")


In [9]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    max_retries=3
)

In [13]:
structured_llm = llm.with_structured_output(CreditFeaturesSchema)

In [14]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert financial analyst. Extract the credit risk features from the provided corporate memo. Ensure all numerical fields are strictly cast as floats."),
    ("user", "{document}")
])

In [15]:
extraction_chain = prompt | structured_llm

sample_10k_text = """
The applicant is requesting a $15,000 loan for debt consolidation over a 36 months term. 
They have an annual income of $85,000 and live in a MORTGAGE home in NY. 
They have been employed for 5 years. Their current interest rate is 12.5% with a $450 monthly installment. 
The loan grade is B, sub-grade B2. Their verification status is Verified. 
Their debt-to-income ratio is 18.5. They have 12 open accounts, 0 public records, 
a revolving balance of $12,400, a utilization rate of 45.2, 22 total accounts, 
2 mortgage accounts, and 0 bankruptcies.
"""

In [16]:
extracted_record = extraction_chain.invoke({"document": sample_10k_text})
payload = extracted_record.model_dump()
print("Extracted Payload:\n", payload)

Extracted Payload:
 {'loan_amnt': 15000.0, 'term': ' 36 months', 'int_rate': 12.5, 'installment': 450.0, 'grade': 'B', 'sub_grade': 'B2', 'emp_length': '5 years', 'home_ownership': 'MORTGAGE', 'annual_inc': 85000.0, 'verification_status': 'Verified', 'purpose': 'debt consolidation', 'addr_state': 'NY', 'dti': 18.5, 'open_acc': 12.0, 'pub_rec': 0.0, 'revol_bal': 12400.0, 'revol_util': 45.2, 'total_acc': 22.0, 'mort_acc': 2.0, 'pub_rec_bankruptcies': 0.0}


In [17]:
import httpx
try:
    response = httpx.post("http://localhost:8000/predict_json", json=payload, timeout=10.0)
    print("\nCredit Risk API Response:\n", response.json())
except httpx.ConnectError:
    print("\nAPI connection skipped: Credit Risk API is not currently running locally on port 8000.")


Credit Risk API Response:
 {'predicted_default': 0}
